In [14]:
from scenario_access import load_scenario

data=load_scenario(32,"",False)

********************************************************
Loading Scenario32 dataset from 


C:\Users\Sedat Cimen\deepsense_positon_non_linear\scenario_access.py:92: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  scenario_df.interpolate(method='linear', axis=0, limit_direction='both',inplace=True)


In [37]:
data.head()

,unit2_spd_over_grnd_kmph,unit2_altitude,unit2_geo_sep,unit2_mode_fix_type,unit2_pdop,unit2_hdop,unit2_vdop,unit2_interpolated_position,unit2_lat,unit2_lon,unit1_beam_index,lat_diff,lon_diff
0,5.820,356.810,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935136,2,0.000000e+00,0.000000
1,5.841,356.812,-27.749,3.0,1.09,0.58,0.92,1.0,33.424312,-111.935134,2,-2.666666e-08,0.000002
2,5.862,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,2,-2.666666e-08,0.000002
3,6.066,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,2,0.000000e+00,0.000000
4,6.351,356.807,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935131,4,-6.000000e-08,0.000002


C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [41]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

# --- 1. GEOMETRİK ÖZELLİKLERİ HESAPLAMA ---
def get_distance(lat1, lon1, lat2, lon2):
    R = 6371000 # metre
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def get_bearing(lat1, lon1, lat2, lon2):
    dLon = np.radians(lon2 - lon1)
    y = np.sin(dLon) * np.cos(np.radians(lat2))
    x = np.cos(np.radians(lat1)) * np.sin(np.radians(lat2)) - \
        np.sin(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.cos(dLon)
    return np.degrees(np.arctan2(y, x))

# Sabit Baz İstasyonu (Unit1) Koordinatları 
u1_lat, u1_lon = 33.4251, -111.9352 

# Yeni sütunları ekleyelim
data['dist_to_bs'] = get_distance(u1_lat, u1_lon, data['unit2_lat'], data['unit2_lon'])
data['bearing'] = get_bearing(u1_lat, u1_lon, data['unit2_lat'], data['unit2_lon'])
data['lat_diff'] = data['unit2_lat'].diff().fillna(0)
data['lon_diff'] = data['unit2_lon'].diff().fillna(0)

# --- 2. VERİ HAZIRLAMA VE PENCERELEME ---
features = [
    'unit2_lat', 'unit2_lon', 'unit2_spd_over_grnd_kmph', 
    'unit2_altitude', 'unit2_pdop', 'dist_to_bs', 'bearing', 'lat_diff', 'lon_diff'
]

X_raw = data[features].values
y_raw = data['unit1_beam_index'].values

# Ölçeklendirme (0-1 arası)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

# Kayan Pencere Fonksiyonu (Son 10 adıma bakarak tahmin)
def create_windows(X, y, window_size=10):
    X_win, y_win = [], []
    for i in range(len(X) - window_size):
        X_win.append(X[i : i + window_size])
        y_win.append(y[i + window_size])
    return np.array(X_win), np.array(y_win)

X_final, y_final = create_windows(X_scaled, y_raw, window_size=10)

# Eğitim ve Test setine ayırma
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

# --- 3. LSTM MODEL MİMARİSİ ---
model = Sequential([
    LSTM(128, input_shape=(10, len(features)), return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(64, activation='softmax') # 64 farklı hüzme (beam) sınıfı
])

model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# --- 4. EĞİTİM ---
print("Eğitim başlıyor...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Eğitim başlıyor...
Epoch 1/50


C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.1740 - loss: 3.1159 - val_accuracy: 0.1628 - val_loss: 3.1297
Epoch 2/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.2554 - loss: 2.4959 - val_accuracy: 0.1054 - val_loss: 2.9294
Epoch 3/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.2911 - loss: 2.3284 - val_accuracy: 0.2295 - val_loss: 2.5359
Epoch 4/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.2860 - loss: 2.2744 - val_accuracy: 0.2450 - val_loss: 2.6891
Epoch 5/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.3074 - loss: 2.2000 - val_accuracy: 0.2527 - val_loss: 2.4319
Epoch 6/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.3205 - loss: 2.1282 - val_accuracy: 0.2186 - val_loss: 2.3509
Epoch 7/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.3116 - loss: 2.1274 - val_accuracy: 0.2341 - val_loss: 2.3596
Epoch 8/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3151 - loss: 2.1442 - val_accuracy: 0.2915 - val_loss: 2.

In [27]:
import tensorflow as tf

# Modelin en iyi 5 tahmini içinde doğru ışın var mı?
top5_acc = tf.keras.metrics.top_k_categorical_accuracy(y_test, model.predict(X_test), k=5)
print(f"Top-5 Doğruluk Oranı: %{np.mean(top5_acc)*100:.2f}")

21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
Top-5 Doğruluk Oranı: %88.37
